In [ ]:
import duckdb
con = duckdb.connect("data/etf.duckdb")

In [ ]:
con.execute("CHECKPOINT;")
con.execute("VACUUM;")
con.close()

In [ ]:
df = con.execute(f"""
select * from silver.observations where family = 'country'
""").df()

In [ ]:
# metric = 'management_expense_ratio'

# display(
# con.execute(f"""
# select 
#     skewness(value)
# from silver.observations
# where metric = '{metric}'
# """).df()
# )

# con.execute(f"""
# select 
#     *
# from silver.observations
# where metric = '{metric}'
# """).df()


### World bank

In [ ]:
import country_converter as coco
import pandas as pd
import duckdb

con = duckdb.connect("data/etf.duckdb")
df = con.execute(
"""
select
    *
from silver.observations
where family in ('country')
"""
).df()

df_target = df[['metric', 'code']].drop_duplicates()

cc = coco.CountryConverter()
df_target["iso3"] = cc.pandas_convert(series=df_target["code"], to="ISO3")
df_target["name_standard"] = cc.pandas_convert(
    series=df_target["code"], to="name_short"
)

TARGET_ISO3 = set(df_target["iso3"].dropna().unique())
print(f"Loaded {len(TARGET_ISO3)} unique target countries/territories across {len(df_target)} rows.")

In [ ]:
SELECTED_WDI = [
    ("NY.GDP.MKTP.CD", "GDP (current US$)", "Economic Output"),
    ("NV.IND.MANF.ZS", "Manufacturing, value added (% of GDP)", "Economic Output"),
    ("NV.SRV.TOTL.ZS", "Services, value added (% of GDP)", "Economic Output"),

    ("SP.POP.TOTL", "Population, total", "Productivity/Output Growth"),
    ("SP.POP.GROW", "Population growth (annual %)", "Productivity/Output Growth"),

    ("SL.UEM.TOTL.ZS", "Unemployment, total (% of total labor force) (modeled ILO estimate)", "Gaps in Wealth"),
    ("SL.TLF.ACTI.ZS", "Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)", "Gaps in Wealth"),

    ("BG.GSR.NFSV.GD.ZS", "Trade in services (% of GDP)", "Trade"),
    ("NE.EXP.GNFS.CD", "Exports of goods and services (current US$)", "Trade"),
    ("NE.IMP.GNFS.CD", "Imports of goods and services (current US$)", "Trade"),

    ("BX.GSR.CCIS.ZS", "ICT service exports (% of service exports, BoP)", "Innovation & Tech"),

    ("GOV_WGI_RL_EST", "Rule of Law - Governance estimate (approx. -2.5 to +2.5)", "Rule of Law/Governance"),
]

catalog = pd.read_csv('dalio_external_data/indicator_catalog.csv')
wdi = catalog[catalog['Source'].str.startswith('World Bank WDI')]
wdi_indicators = wdi.copy()
# wdi_indicators = wdi[wdi['code'].isin([e[0] for e in SELECTED_WDI])]
wdi_indicators = wdi_indicators[['code', 'label']].set_index('code').to_dict()['label']

In [ ]:
# Fetch indicators
import concurrent.futures
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import numpy as np

# 1. Setup Country Code Resolvers
iso_map = dict(zip(cc.data["ISO2"].dropna(), cc.data["ISO3"].dropna()))
iso_map.update({"EL": "GRC", "UK": "GBR"})

def resolve_iso3(val):
    if not val:
        return None
    c = str(val).strip().upper()
    return c if (len(c) == 3 and c in TARGET_ISO3) else iso_map.get(c)

# 2. Resilient HTTP Session with Connection Pooling & Exponential Backoff
session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
session.mount("https://", adapter)
session.mount("http://", adapter)

# 3. Indicator Fetch Worker
def fetch_indicator(code):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}"
    params = {"format": "json", "per_page": 1000, "date": "2020:2026"}
    rows, page, pages = [], 1, 1

    try:
        while page <= pages:
            # (connect_timeout=10, read_timeout=60)
            resp = session.get(url, params={**params, "page": page}, timeout=(10, 60))
            if resp.status_code != 200:
                return code, [], f"HTTP {resp.status_code}"
            
            data = resp.json()
            if not isinstance(data, list) or len(data) < 2 or not data[1]:
                break
                
            pages = int(data[0]["pages"])
            rows.extend(data[1])
            page += 1

        records = []
        for r in rows:
            iso = resolve_iso3(r.get("countryiso3code") or (r.get("country") or {}).get("id"))
            if iso in TARGET_ISO3 and r.get("value") is not None:
                records.append({
                    "code": code,
                    "indicator_label": wdi_indicators.get(code, code),
                    "iso3": iso,
                    "year": int(r["date"]),
                    "value": float(r["value"])
                })
        return code, records, None

    except Exception as e:
        return code, [], str(e)


# 4. Incremental / Resumable State Handling
try:
    panel = pd.read_csv('panel.csv')
except Exception:
    pass

if "panel" not in globals() or not isinstance(panel, pd.DataFrame):
    panel = pd.DataFrame(columns=["code", "indicator_label", "iso3", "year", "value"])

already_fetched = set(panel["code"].unique())
pending_indicators = [code for code in wdi_indicators.keys() if code not in already_fetched]

print(f"Total indicators in catalog: {len(wdi_indicators)}")
print(f"Already in panel: {len(already_fetched)} | Pending fetch: {len(pending_indicators)}")

# 5. Fetch Pending Indicators Concurrently
failed_indicators = {}
new_records = []

if pending_indicators:
    # 4 workers strikes the best balance between speed and avoiding WB rate-limiting
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(fetch_indicator, code): code for code in pending_indicators}
        
        for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
            code, records, error = future.result()
            if error:
                print(f"[{i}/{len(pending_indicators)}] ⚠️ Failed: {code} ({error})")
                failed_indicators[code] = error
            else:
                new_records.extend(records)
                print(f"[{i}/{len(pending_indicators)}] ✅ Fetched: {code} ({len(records)} records)")

    # Accumulate into existing panel and persist deduplicated records
    if new_records:
        panel = pd.concat([panel, pd.DataFrame(new_records)], ignore_index=True)
        panel.drop_duplicates(subset=["code", "iso3", "year"], inplace=True)

print(f"\nFetch complete. Total rows in panel: {len(panel):,}")
if failed_indicators:
    print(f"Indicators to retry later ({len(failed_indicators)}): {list(failed_indicators.keys())}")

del new_records, failed_indicators, already_fetched, pending_indicators

In [ ]:
# derive indicators
import numpy as np
import pandas as pd

# 1. Base Indicators & Derived Definitions
ind = {
    "POP": "SP.POP.TOTL",
    "GDP": "NY.GDP.MKTP.CD",
    "EXP": "NE.EXP.GNFS.CD",
    "IMP": "NE.IMP.GNFS.CD",
    "LFPR": "SL.TLF.ACTI.ZS",
    "UEM": "SL.UEM.TOTL.ZS",
}

DERIVED = {
    "D.TRADE.BAL": (
        "Goods & services trade balance (US$)",
        "Trade",
        lambda w: w[ind["EXP"]] - w[ind["IMP"]],
    ),
    "D.TRADE.BAL.GDP": (
        "Goods & services trade balance (% of GDP)",
        "Trade",
        lambda w: (w[ind["EXP"]] - w[ind["IMP"]]) / w[ind["GDP"]].replace(0, np.nan) * 100.0,
    ),
    "D.EMPPOP.RATIO": (
        "Employment to Population Ratio",
        "Wellness & Inequality",
        lambda w: w[ind["LFPR"]] * (100.0 - w[ind["UEM"]]) / 100.0,
    ),
}

# 2. Derive Features Safely
panel = panel[~panel["code"].isin(DERIVED.keys())].copy()
panel["value"] = pd.to_numeric(panel["value"], errors="coerce")

base = panel.pivot_table(index=["iso3", "year"], columns="code", values="value").sort_index()

derived_frames = []
for d_code, (label, cat, fn) in DERIVED.items():
    try:
        s = (
            fn(base)
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .rename("value")
            .reset_index()
        )
        s["code"] = d_code
        s["indicator_label"] = label
        derived_frames.append(s[["code", "indicator_label", "iso3", "year", "value"]])
    except KeyError as e:
        print(f"Skipping derived indicator {d_code}: missing input series {e}")

if derived_frames:
    panel = pd.concat([panel, pd.concat(derived_frames, ignore_index=True)], ignore_index=True)

for code, (label, cat, _) in DERIVED.items():
    wdi_indicators[code] = label
    if "category_map" in globals():
        category_map[code] = cat

# ---------------------------------------------------------------- config
CFG = dict(
    min_coverage=0.76,     # share of ACTIVE countries with any value
    min_recent_cov=0.7,    # share of active countries with a value in last `recent_years`
    recent_years=3.0,
    min_recent_grid=0.5,   # share of the active country x recent-year grid filled (change tier only)
    min_grid=0.55,          # share of the active country x year grid filled (change tier only)
    min_recent_obs=0,
    min_obs=0,             # mean obs per country (change tier only)
    min_distinct=1,
    max_mode_share=1,    # one value > 50% of observations => degenerate
    max_zero_pct=1,
    dyn_structural=0.1,    # dynamism below -> "structural"
    dyn_dynamic=0.3,       # dynamism above -> "dynamic"
)
EXCLUDE = {ind["EXP"], ind["IMP"], ind["UEM"]}  # inputs to derived series

In [ ]:
# Analyze indicators
import re

# ------------------------------------------------------ evaluation panel
eval_panel = panel[~panel["code"].isin(EXCLUDE)].copy()
eval_panel["value"] = pd.to_numeric(eval_panel["value"], errors="coerce")
eval_panel = eval_panel.dropna(subset=["value"])

active_iso = eval_panel["iso3"].unique()
n_active = len(active_iso)                     # ~106, not 109
years = sorted(eval_panel["year"].unique())
n_years = len(years)
recent_cut = max(years) - CFG["recent_years"] + 1
n_recent_years = sum(y >= recent_cut for y in years)   # actual years in window

def to_scale(v):
    """Log positive, right-skewed series; leave everything else alone."""
    if len(v) > 2 and (v > 0).all() and v.skew() > 1:
        return np.log(v)
    return v

eval_panel["t"] = eval_panel.groupby("code")["value"].transform(to_scale)

# ------------------------------------------------------------ diagnostics
def diagnose(g):
    by = g.groupby("iso3")["t"]
    means = by.mean()
    between_var = means.var()
    within_var = by.var().mean()               # NaN if every country has 1 obs
    total_sd = g["t"].std()
    between_sd = np.sqrt(between_var) if pd.notna(between_var) else np.nan
    within_sd = np.sqrt(within_var) if pd.notna(within_var) else np.nan
    denom = (between_var or 0) + (within_var if pd.notna(within_var) else 0)
    recent = g[g["year"] >= recent_cut]
    recent_obs = recent.groupby("iso3").size().mean() if len(recent) else np.nan

    return pd.Series({
        "coverage": g["iso3"].nunique() / n_active,
        "recent_cov": recent["iso3"].nunique() / n_active,
        "grid_cov": len(g) / (n_active * n_years),
        "recent_grid_cov": len(recent) / (n_active * n_recent_years),
        "obs_per_country": by.size().mean(),
        "recent_obs_per_country": recent_obs,
        "n_distinct": g["value"].nunique(),
        "mode_share": g["value"].value_counts(normalize=True).iloc[0],
        "zero_pct": (g["value"] == 0).mean(),
        "logged": bool((g["t"] != g["value"]).any()),
        "total_sd": total_sd,
        "between_sd": between_sd,
        "within_sd": within_sd,
        "dynamism": within_sd / (between_sd + 1e-9) if pd.notna(within_sd) else np.nan,
        "icc": between_var / denom if denom > 0 else np.nan,
        "skew_raw": g["value"].skew() if len(g) > 2 else np.nan,
    })

diag = eval_panel.groupby("code").apply(diagnose, include_groups=False)
diag.insert(0, "indicator", diag.index.map(wdi_indicators))
if "category_map" in globals():
    diag.insert(1, "category", diag.index.map(category_map))

# ------------------------------------------------------------------ gates
gates = pd.DataFrame({
    "coverage":      diag["coverage"]   >= CFG["min_coverage"],
    "recent":        diag["recent_cov"] >= CFG["min_recent_cov"],
    "non_degenerate": (
        (diag["n_distinct"]  >= CFG["min_distinct"]) &
        (diag["mode_share"]  <= CFG["max_mode_share"]) &
        (diag["zero_pct"]    <= CFG["max_zero_pct"]) &
        (diag["total_sd"]    >  0)
    ),
    "recent_grid":   diag["recent_grid_cov"]        >= CFG["min_recent_grid"],
    "recent_obs":    diag["recent_obs_per_country"] >= CFG["min_recent_obs"],
    "grid":          diag["grid_cov"]               >= CFG["min_grid"],
    "obs":           diag["obs_per_country"]        >= CFG["min_obs"],
    "moves":         diag["within_sd"]              >  0,
})

LEVEL_GATES  = ["coverage", "recent", "non_degenerate"]
CHANGE_GATES = LEVEL_GATES + ["recent_grid", "recent_obs", "grid", "obs", "moves"]

diag["level_ok"]  = gates[LEVEL_GATES].all(axis=1)
diag["change_ok"] = gates[CHANGE_GATES].all(axis=1)

# --------------------------------------------------------- tags (not filters)
diag["role"] = pd.cut(
    diag["dynamism"],
    bins=[-np.inf, CFG["dyn_structural"], CFG["dyn_dynamic"], np.inf],
    labels=["structural", "mixed", "dynamic"],
)
is_scale = (
    diag["indicator"].str.contains(r"\((?:current|constant)[^)]*\)", flags=re.I, regex=True)
    & ~diag["indicator"].str.contains(r"per capita|% of|per \d", flags=re.I, regex=True)
)
diag["scale_dependent"] = is_scale

# ----------------------------------------------------------------- funnel
print(f"Active countries: {n_active} | years: {years[0]}-{years[-1]} | recent cut: {recent_cut} ({n_recent_years} yrs)")
print(f"{len(diag)} indicators evaluated\n")

print("Gate fail counts (independent):")
print((~gates).sum().to_string(), "\n")

print("Cumulative funnel (change tier):")
remaining = pd.Series(True, index=gates.index)
for name in CHANGE_GATES:
    remaining &= gates[name]
    print(f"  after {name:<15} {remaining.sum():>5}")

print(f"\nlevel_ok : {diag['level_ok'].sum()}")
print(f"change_ok: {diag['change_ok'].sum()}")
print(f"level-only (sparse but usable): {(diag['level_ok'] & ~diag['change_ok']).sum()}")
print("\nRole among level_ok:\n", diag.loc[diag["level_ok"], "role"].value_counts().to_string())
print(f"\nScale-dependent among level_ok: {diag.loc[diag['level_ok'], 'scale_dependent'].sum()}")

In [ ]:
diag.loc[[
        "VC.IHR.PSRC.P5",
        "EN.ATM.PM25.MC.M3",
        "GB.XPD.RSDV.GD.ZS",
        "SP.POP.SCIE.RD.P6",
        "CM.MKT.LCAP.GD.ZS",
        "SH.STA.SUIC.P5",
    ]]

In [ ]:
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_histograms(df, nbins=30, cols_per_row=3):
    # Select only numeric columns
    numeric_cols = df.select_dtypes(include='number').columns
    
    if len(numeric_cols) == 0:
        print("No numeric columns found.")
        return

    # Calculate layout grid dimensions
    n_rows = math.ceil(len(numeric_cols) / cols_per_row)
    
    # Create subplots
    fig = make_subplots(
        rows=n_rows,
        cols=cols_per_row,
        subplot_titles=numeric_cols
    )
    
    # Add each histogram to the grid
    for i, col in enumerate(numeric_cols):
        row = (i // cols_per_row) + 1
        col_idx = (i % cols_per_row) + 1
        
        fig.add_trace(
            go.Histogram(x=df[col], nbinsx=nbins, name=col),
            row=row,
            col=col_idx
        )
    
    # Adjust sizing & styling
    fig.update_layout(
        height=280 * n_rows,
        width=320 * cols_per_row,
        showlegend=False,
        title_text="Histograms of Numeric Columns",
        bargap=0.05
    )
    
    fig.show()

# Example usage:
# plot_histograms(diag, nbins=100)
plot_histograms(diag[diag['change_ok']], nbins=100)

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
import numpy as np
import pandas as pd

corr_matrix = wide.corr(method="spearman").fillna(0)
dist_matrix = (1.0 - corr_matrix.abs()).to_numpy(copy=True)

dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
np.fill_diagonal(dist_matrix, 0.0)
dist_matrix = np.clip(dist_matrix, 0.0, 2.0)

condensed_dist = squareform(dist_matrix)
Z = linkage(condensed_dist, method="complete")

DISTANCE_THRESHOLD = 0.1
clusters = fcluster(Z, t=DISTANCE_THRESHOLD, criterion="distance")

cluster_df = pd.DataFrame(
    {
        "code": wide.columns,
        "cluster": clusters,
        "quality": quality.reindex(wide.columns).values,
    }
)

lean_kept = (
    cluster_df.sort_values("quality", ascending=False)
    .groupby("cluster")
    .first()["code"]
    .tolist()
)

cluster_df

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = wide.dropna().copy()
X = (X - X.mean()) / X.std()

def prune_vif(df, threshold=5.0):
    cols = list(df.columns)
    while True:
        vifs = [variance_inflation_factor(df[cols].values, i) for i in range(len(cols))]
        max_vif = max(vifs)
        if max_vif <= threshold:
            break
        drop_col = cols[np.argmax(vifs)]
        print(drop_col, max_vif)
        cols.remove(drop_col)
    return cols

orthogonal_subset = prune_vif(X, threshold=1.0)
len(orthogonal_subset)

In [ ]:
import duckdb
import orjson
import zstandard as zstd

# con = duckdb.connect("data/etf.duckdb")

payload = con.execute("""
    SELECT payload
    FROM bronze.payload_blobs
    WHERE hash = 3792843352904083660
""").fetchone()[0]

_DECOMPRESSOR = zstd.ZstdDecompressor()

def decompress_payload(compressed: bytes):
    canonical = _DECOMPRESSOR.decompress(compressed)
    return orjson.loads(canonical)

result = decompress_payload(payload)

In [ ]:
df_db_size = con.execute("""
    SELECT 
        database_size,
        block_size,
        free_blocks,
        total_blocks,
        wal_size,
        memory_limit,
        -- Cast calculation to BIGINT for format_bytes
        format_bytes(CAST(total_blocks AS BIGINT) * CAST(block_size AS BIGINT)) AS total_allocated_blocks,
        format_bytes(CAST(free_blocks AS BIGINT) * CAST(block_size AS BIGINT)) AS reclaimable_space
    FROM pragma_database_size()
""").df()

df_db_size.to_csv('test.csv', index=False)

## IBKR CLIENT TEST


In [ ]:
# Config and imports
import asyncio
from datetime import datetime, date
import math
from pathlib import Path
import duckdb
import pandas as pd
from tqdm.auto import tqdm
from ib_async import IB, Contract, util
import logging

util.logToConsole(level=logging.ERROR)

DB_PATH = Path("data/etf.duckdb")

HOST = "127.0.0.1"
PORT = 4001           # 7497 (TWS Paper), 7496 (TWS Live), 4002/4001 (Gateway)
CLIENT_ID = 10        # Ensure this is unique across your connections

DEFAULT_DURATION = "5 Y"  # Initial fetch length ("1 Y", "5 Y", "20 Y", etc.)
WHAT_TO_SHOW = "TRADES"   # "TRADES" or "MIDPOINT"
RATE_LIMIT_DELAY = 0.25   # Delay (seconds) between calls to avoid pacing violations

In [ ]:
# Create the client and connect (top-level await in Jupyter)
ib = IB()
await ib.connectAsync(host="127.0.0.1", port=4001, clientId=9, timeout=5)

In [ ]:
# Query product metadata from bronze.products in DuckDB
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    products_df = con.execute("""
        SELECT 
            *
        FROM bronze.products
    """).df()

# print(f"Loaded {len(products_df)} products from {DB_PATH}")
products_df

In [ ]:
# Second semaphore test
import asyncio
from tqdm.asyncio import tqdm_asyncio

# products = list(zip(products_df['product_id'], products_df['exchange_id']))
# conids = products_df['product_id'].to_list()[:2]
conids = [8335, 407137443]

sem = asyncio.Semaphore(1)
async def validate_contract(cid):
    async with sem:
        try:
            details = await ib.reqContractDetailsAsync(
                Contract(conId=int(cid))
            )
            return details if details else None
        except Exception:
            return None

tasks = [validate_contract(cid) for cid in conids]
results = await tqdm_asyncio.gather(*tasks)
results

In [ ]:
# contract = Contract(secType='STK', conId=407137443, symbol='TNAEN', exchange='MEXI')
contract = results[0][0].contract

bars = await ib.reqHistoricalDataAsync(
    contract,
    endDateTime='',
    durationStr='365 D',
    barSizeSetting='1 day',
    whatToShow='ADJUSTED_LAST',
    useRTH=True,
    formatDate=1
)

In [ ]:
if ib.isConnected():
    ib.disconnect()
    print("Disconnected from IBKR.")

## Database table memory

In [ ]:
import duckdb
import pandas as pd

def get_table_storage_stats(db_path: str):
    con = duckdb.connect(db_path, read_only=True)
    
    # 1. Fetch all physical tables across custom schemas
    tables_df = con.sql("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        WHERE table_type = 'BASE TABLE'
          AND table_schema NOT IN ('information_schema', 'pg_catalog')
        ORDER BY table_schema, table_name
    """).df()

    results = []

    # 2. Iterate and sum block allocation per table
    for _, row in tables_df.iterrows():
        schema = row['table_schema']
        table = row['table_name']
        full_table_name = f'"{schema}"."{table}"'
        
        try:
            # Query segment storage info for the table
            storage_df = con.sql(f"PRAGMA storage_info('{full_table_name}')").df()
            
            # Each distinct row group segment maps to DuckDB's 256 KiB block allocation unit
            allocated_blocks = storage_df['segment_type'].notna().sum()
            allocated_bytes = allocated_blocks * 262144  # 256 KiB = 262,144 bytes

            # Fetch approximate row count
            row_count = con.sql(f"SELECT COUNT(*) FROM {full_table_name}").fetchone()[0]

            results.append({
                "Schema": schema,
                "Table": table,
                "Rows": row_count,
                "Allocated Bytes": allocated_bytes,
                "Formatted Size": format_bytes(allocated_bytes)
            })
        except Exception as e:
            results.append({
                "Schema": schema,
                "Table": table,
                "Rows": 0,
                "Allocated Bytes": 0,
                "Formatted Size": f"Error: {str(e)}"
            })

    con.close()
    
    df = pd.DataFrame(results)
    return df

def format_bytes(size):
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if abs(size) < 1024.0:
            return f"{size:3.2f} {unit}"
        size /= 1024.0
    return f"{size:.2f} PiB"


df_stats = get_table_storage_stats("data/etf.duckdb")
df_stats

In [ ]:
df_stats['Formatted Size'].str.replace(' MiB', '').str.strip().astype(float).sum()

## OLD DATA

In [ ]:
import pandas as pd

df = pd.read_csv('data/contract_scraped_25-08.csv')
df

In [ ]:
import ast
import pandas as pd
import numpy as np

debtors = df['debt_type'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
).explode()

debtors = debtors.apply(lambda x:x[0] if pd.notna(x) else np.nan)

In [ ]:
debtors.value_counts()

# Silve Catalog Exploration

In [ ]:
import sqlite3
import pandas as pd
con = sqlite3.connect("data/silver_catalog.sqlite")
pd.read_sql("select * from sqlite_master where type == 'table'", con)